In [ ]:
!pip install -U langchain-google-genai langgraph chromadb tavily-python python-dotenv tiktoken langchain-community langchainhub --quiet

In [ ]:
!pip install python-dotenv langchain-google-genai langgraph chromadb tavily-python tiktoken langchain-community langchainhub

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. Cargar de forma segura las llaves desde el archivo .env
load_dotenv()

if not os.getenv("GOOGLE_API_KEY") or not os.getenv("TAVILY_API_KEY"):
    raise ValueError("¡Faltan configurar tus llaves en el archivo .env!")

print("✅ Credenciales cargadas con éxito.")

# 2. Configurar el "Cerebro" (LLM) - Reemplaza a OpenAI
# Usamos gemini-1.5-flash: es súper rápido, inteligente y gratuito
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2
)

# 3. Configurar el Generador de Vectores (Embeddings) para tu base de datos ChromaDB
embd = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

print("🤖 Modelos de Gemini inicializados correctamente de forma gratuita.")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

# 1. Definir las URLs que utilizaremos como fuente de conocimiento
urls = [
    "https://www.nutmeg.com/nutmegonomics/our-2025-investment-outlook",
    "https://www.nutmeg.com/nutmegonomics/nutmeg-investor-update-december-2024",
    "https://www.nutmeg.com/nutmegonomics/nutmeg-investor-update-november-2024",
    "https://www.nutmeg.com/nutmegonomics/nutmeg-investor-update-october-2024"
]

# 2. Descargar y cargar los documentos de la web
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# 3. Fragmentar el texto en bloques de 500 caracteres
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)

# 4. Almacenar los fragmentos en nuestra base de datos vectorial ChromaDB
# Recuerda que 'embd' es la configuración que creamos para GoogleGenAI
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
)
embedding=embd,
retriever = vectorstore.as_retriever()

print(f"✅ Base de datos vectorial configurada exitosamente con {len(doc_splits)} fragmentos.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

# Definimos el formato de salida esperado
class RouteQuery(BaseModel):
    """Enruta una pregunta del usuario a la fuente de datos más relevante."""
    datasource: Literal["vectorstore", "web_search"] = Field(
        ...,
        description="Dada una pregunta de usuario, elige enviarla a web_search o a vectorstore.",
    )

# Estructuramos a Gemini para que responda estrictamente bajo este formato Pydantic
structured_llm_router = llm.with_structured_output(RouteQuery)

# El prompt del sistema con las reglas
ROUTER_PROMPT = """Eres un experto en enrutar preguntas de usuarios a un vectorstore o a una búsqueda web.
El vectorstore contiene documentos relacionados a las proyecciones de inversión de Nutmeg para el 2025 o actualizaciones de finanzas de fin de año de la empresa.
Usa el vectorstore exclusivamente para preguntas sobre estos temas corporativos financieros de Nutmeg. De lo contrario, usa web_search."""

route_prompt = ChatPromptTemplate.from_messages([
    ("system", ROUTER_PROMPT),
    ("human", "{question}"),
])

question_router = route_prompt | structured_llm_router

# --- ZONA DE PRUEBA DEL ENRUTADOR ---
# Prueba 1: Debería ir al vectorstore
print(question_router.invoke({"question": "What is the 2025 investment outlook for nutmeg?"}))

# Prueba 2: Debería ir a web_search
print(question_router.invoke({"question": "Who won the last football world cup?"}))

In [ ]:
# --- EVALUADOR DE RELEVANCIA DE DOCUMENTOS ---
class GradeDocuments(BaseModel):
    binary_score: str = Field(description="¿Los documentos son relevantes para la pregunta?, 'yes' o 'no'")

structured_llm_grader = llm.with_structured_output(GradeDocuments)

BINARY_PROMPT = """Eres un evaluador que califica la relevancia de un documento recuperado para una pregunta de usuario.
Si el documento contiene palabras clave o significado semántico relacionado a la pregunta, califícalo como relevante con 'yes'. De lo contrario, pon 'no'."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", BINARY_PROMPT),
    ("human", "Documento recuperado: \n\n {document} \n\n Pregunta de usuario: {question}"),
])
doc_grader = grade_prompt | structured_llm_grader


# --- EVALUADOR DE ALUCINACIONES ---
class GradeHallucinations(BaseModel):
    binary_score: str = Field(description="¿La respuesta está fundamentada en los hechos proporcionados?, 'yes' o 'no'")

structured_llm_hallucination_grader = llm.with_structured_output(GradeHallucinations)

HALLUCINATION_PROMPT = """Eres un evaluador que califica si una respuesta de IA está basada/soportada en un conjunto de hechos recuperados.
Da una calificación binaria de 'yes' o 'no'. 'yes' significa que la respuesta está fundamentada en los hechos y no inventa nada."""

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", HALLUCINATION_PROMPT),
    ("human", "Conjunto de hechos: \n\n {documents} \n\n Respuesta de la IA: {generation}"),
])
hallucination_grader = hallucination_prompt | structured_llm_hallucination_grader

print("✅ Evaluadores de calidad inicializados con Gemini.")

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import Tavily some_search_tool
from langchain_community.utilities import TavilySearchAPIWrapper

# Definimos la memoria/estado del grafo de LangGraph
class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]

# Inicializamos la herramienta de búsqueda de Tavily
from langchain_community.tools.tavily_search import TavilyTextInput, TavilySearchResults
web_search_tool = TavilySearchResults(k=3) 

print("✅ Estado del grafo y herramienta Tavily configurados.")

In [ ]:
#Construir el grafo de LangGraph
from langchain.schema import Document

# NODO 1: Recuperar documentos de ChromaDB
def retrieve(state):
    print("--- RETRIEVE FROM VECTORSTORE ---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

# NODO 2: Generar la respuesta usando Gemini
def generate(state):
    print("--- GENERATE ANSWER ---")
    question = state["question"]
    documents = state["documents"]
    
    # Usamos un prompt rápido de RAG para Gemini
    from langchain_core.prompts import PromptTemplate
    prompt = PromptTemplate.from_template(
        "Answer the question based only on the following context:\n{context}\n\nQuestion: {question}"
    )
    rag_chain = prompt | llm
    
    # Juntamos los textos de los documentos
    context = "\n\n".join([doc.page_content for doc in documents])
    generation = rag_chain.invoke({"context": context, "question": question})
    
    return {"documents": documents, "question": question, "generation": generation.content}

# NODO 3: Buscar en internet con Tavily
def web_search(state):
    print("--- WEB SEARCH WITH TAVILY ---")
    question = state["question"]
    
    # Ejecutamos la búsqueda web
    docs = web_search_tool.invoke({"query": question})
    
    # Convertimos los resultados de Tavily en el formato adecuado
    web_results = "\n".join([d["content"] for d in docs])
    web_results = Document(page_content=web_results)
    
    return {"documents": [web_results], "question": question}

In [ ]:
#Bordes condicionales
# DECISIÓN 1: ¿Ir a la base de datos o a internet?
def route_question(state):
    print("--- ROUTE QUESTION ---")
    question = state["question"]
    source = question_router.invoke({"question": question})
    
    if source.datasource == "web_search":
        print("-> ROUTE: WEB SEARCH")
        return "web_search"
    else:
        print("-> ROUTE: VECTORSTORE")
        return "retrieve"

# DECISIÓN 2: ¿Los documentos sirven o necesitamos ir a internet?
def decide_to_generate(state):
    print("--- ASSESS DOCUMENTS ---")
    question = state["question"]
    documents = state["documents"]
    
    # Evaluamos el primer documento recuperado
    score = doc_grader.invoke({"question": question, "document": documents[0].page_content})
    
    if score.binary_score == "yes":
        print("-> DECISION: DOCUMENTS ARE RELEVANT -> GENERATE")
        return "generate"
    else:
        print("-> DECISION: NOT RELEVANT -> GO TO WEB SEARCH")
        return "web_search"

# DECISIÓN 3: ¿La respuesta es una alucinación o es válida?
def grade_generation_v_documents(state):
    print("--- CHECK FOR HALLUCINATIONS ---")
    documents = state["documents"]
    generation = state["generation"]
    
    score = hallucination_grader.invoke({"documents": documents, "generation": generation})
    
    if score.binary_score == "yes":
        print("-> DECISION: ANSWER IS GROUNDED IN FACTS -> FINISH")
        return "useful"
    else:
        print("-> DECISION: HALLUCINATION DETECTED -> RE-GENERATE")
        return "not useful"

In [ ]:
#Construimos el grafo de LangGraph
from langgraph.graph import END, StateGraph, START

# 1. Creamos el flujo basado en el estado que definimos antes
workflow = StateGraph(GraphState)

# 2. Añadimos los Nodos principales
workflow.add_node("retrieve", retrieve)
workflow.add_node("web_search", web_search)
workflow.add_node("generate", generate)

# 3. Construimos el mapa de conexiones (Lógica de control)
workflow.add_conditional_edges(
    START,
    route_question,
    {
        "web_search": "web_search",
        "retrieve": "retrieve",
    },
)

workflow.add_conditional_edges(
    "retrieve",
    decide_to_generate,
    {
        "web_search": "web_search",
        "generate": "generate",
    },
)

workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents,
    {
        "not useful": "generate", # Si alucina, vuelve a intentar generar
        "useful": END,            # Si es correcta, termina el proceso
    },
)

workflow.add_edge("web_search", "generate")

# 4. Compilamos el agente listo para usar
app = workflow.compile()
print("🎯 ¡Grafo de LangGraph compilado con éxito!")

In [ ]:
#PRUEBA FINAL DEL AGENTE ADAPTIVO 1
inputs = {"question": "What is the 2025 investment outlook for nutmeg?"}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Node '{key}' completed.")
print("\n🤖 Respuesta Final:\n", value["generation"])

In [ ]:
#Prueba con Gemini
inputs = {"question": "What is the current capital of France?"}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Node '{key}' completed.")
print("\n🤖 Respuesta Final:\n", value["generation"])